# GTEx_V11_GTEx_Midterm_Research

**Research question (RQ):** How accurately can gene expression in one human brain region predict expression in another in unseen GTEx donors, and how does predictive performance vary across region pairs?

**Dataset:** GTEx V11 gene-read-count matrices for 12 human brain regions (the supplied ZIP also contains cervical spinal cord, which is excluded). The donor is the independent unit. No external disease datasets are used.

**Reproduce:** Open this notebook in [Google Colab](https://colab.research.google.com/) and choose **Runtime → Run all**. Section 2 downloads `gene_reads_GTEx.zip` from GitHub Release `ellamichele/BMEN600Group23`, tag `v1.0-data`. The GitHub Release must be public for reviewers without credentials. Sections 3–5 load and prepare data, make Figure 1, and generate Table 2 and Figure 5. Run time depends on the available Colab resources.

**Midterm outputs:** Figure 1 (available donors per region), Table 2 (held-out prediction metrics), and Figure 5 (median gene-wise test R² with donor-bootstrap 95% intervals). These appear in Colab and are saved under `/content/gtex_results`.


## 1. Settings and imports


In [ ]:
from pathlib import Path
import gzip, hashlib, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from IPython.display import display

SEED = 42
RUN_ALL_PAIRS = False  # False: six illustrative directions; True: all eligible directions
MIN_PAIRED_DONORS = 60
N_SOURCE_GENES = 500
N_TARGET_GENES = 1000
ALPHAS = [1., 10., 100., 1000.]
N_BOOTSTRAPS = 1000

OUT = Path('/content/gtex_results')
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

ILLUSTRATIVE_PAIRS = [
    ('caudate_basal_ganglia', 'putamen_basal_ganglia'),
    ('putamen_basal_ganglia', 'caudate_basal_ganglia'),
    ('cortex', 'frontal_cortex_ba9'),
    ('frontal_cortex_ba9', 'cortex'),
    ('cerebellum', 'cerebellar_hemisphere'),
    ('cerebellar_hemisphere', 'cerebellum'),
]


## 2. Download GTEx V11 data
Downloads and checks `gene_reads_GTEx.zip` from the public GitHub Release.


In [ ]:
from pathlib import Path
import requests
import zipfile
import shutil

# Public GitHub Release containing the GTEx input data.
GITHUB_OWNER = "ellamichele"
GITHUB_REPO = "BMEN600Group23"
RELEASE_TAG = "v1.0-data"
ZIP_NAME = "gene_reads_GTEx.zip"

DATA_URL = (
    f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}/"
    f"releases/download/{RELEASE_TAG}/{ZIP_NAME}"
)
DATA_DIR = Path("/content/gtex_input")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path("/content") / ZIP_NAME

print(f"Downloading GTEx V11 data from: {DATA_URL}")
try:
    with requests.get(DATA_URL, stream=True, timeout=(30, 180),
                      headers={"User-Agent": "GTEx-BMEN600-reproducibility-notebook"}) as response:
        response.raise_for_status()
        with ZIP_PATH.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
except requests.RequestException as exc:
    raise RuntimeError(
        "GitHub dataset download failed. Confirm that the repository and Release "
        "are public and that the Release contains the asset gene_reads_GTEx.zip. "
        f"Attempted URL: {DATA_URL}"
    ) from exc

# Extract expected data files only. Flatten optional directories inside ZIP.
# Do not extract arbitrary paths, and reject duplicate region filenames.
try:
    with zipfile.ZipFile(ZIP_PATH) as archive:
        members = [m for m in archive.infolist()
                   if not m.is_dir() and Path(m.filename).name.endswith(".gct.gz")]
        names = [Path(m.filename).name for m in members]
        if len(names) != len(set(names)):
            raise ValueError("ZIP contains duplicate .gct.gz filenames.")
        if len(names) != 13:
            raise ValueError(f"Expected 13 .gct.gz files in ZIP, found {len(names)}")
        unexpected = [name for name in names if not
                      (name.startswith("gene_reads_adult_gtex_v11_brain_") and
                       name.endswith(".gct.gz"))]
        if unexpected:
            raise ValueError(
                "Unexpected dataset filename(s): " + ", ".join(unexpected)
                + ". Ensure the files retain their original names (no '(1)' suffixes)."
            )
        for m in members:
            destination = DATA_DIR / Path(m.filename).name
            with archive.open(m) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
except (zipfile.BadZipFile, OSError, ValueError) as exc:
    raise RuntimeError("Unable to validate/extract GTEx release ZIP: " + str(exc)) from exc

paths = sorted(DATA_DIR.glob("gene_reads_adult_gtex_v11_brain_*.gct.gz"))
assert len(paths) == 13, f"Expected 13 region files after extraction, found {len(paths)}"
for path in paths:
    with path.open("rb") as f:
        if f.read(2) != b"\x1f\x8b":
            raise ValueError(f"Not a valid gzip file header: {path.name}")
print(f"Dataset ready: {len(paths)} region files downloaded and identified.")
print("Run Section 3 to load, validate, and normalize the count matrices.")

## 3. Load and normalize GTEx gene counts
Validate GCT format, calculate log₂(CPM + 1), and match samples to GTEx donors. A donor is counted once per region.


In [ ]:
def load_region(path):
    with gzip.open(path, 'rt') as f:
        version = f.readline().strip()
        n_genes, n_samples = map(int, f.readline().split())
    if version != '#1.2':
        raise ValueError(f'Unsupported GCT version: {version}')
    table = pd.read_csv(path, sep='\t', skiprows=2)
    if table.shape != (n_genes, n_samples + 2):
        raise ValueError(f'Dimension mismatch: {path.name}')
    if table['Name'].duplicated().any():
        raise ValueError('Duplicate gene IDs')
    symbols = table.set_index('Name')['Description']
    counts = table.iloc[:, 2:].astype('float32')
    counts.index = table['Name']
    if not np.isfinite(counts.to_numpy()).all() or (counts.to_numpy() < 0).any():
        raise ValueError('Counts must be finite and nonnegative')
    libraries = counts.sum(axis=0)
    if (libraries <= 0).any():
        raise ValueError('Zero-count sample found')
    donors = []
    for sample in counts.columns:
        parts = sample.split('-')
        if len(parts) < 3 or parts[0] != 'GTEX':
            raise ValueError(f'Unexpected sample ID: {sample}')
        donors.append('-'.join(parts[:2]))
    logcpm = np.log2(counts.div(libraries, axis=1) * 1e6 + 1).T
    logcpm.index = donors
    matrix = logcpm.groupby(level=0, sort=True).mean().astype('float32')
    record = dict(region=path.name.removeprefix('gene_reads_adult_gtex_v11_brain_').removesuffix('.gct.gz'),
                  genes=n_genes, samples=n_samples, donors=len(matrix),
                  extra_samples=n_samples-len(matrix), median_library_reads=float(libraries.median()),
                  sha256=hashlib.sha256(path.read_bytes()).hexdigest())
    return matrix, symbols, record

regions, records = {}, []
for path in paths:
    matrix, gene_symbols, record = load_region(path)
    regions[record['region']] = matrix
    records.append(record)
    print(record['region'], matrix.shape)
summary = pd.DataFrame(records)
display(summary.drop(columns='sha256'))
active = sorted(r for r in regions if not r.startswith('spinal_cord'))


## 4. Figure 1 — Available donors


In [ ]:
# Figure 1. Number of unique GTEx donors with samples from each supplied region.
fig, ax = plt.subplots(figsize=(10, 6))
s = summary.sort_values('donors')
ax.barh(s.region.str.replace('_', ' '), s.donors, color='#357da8')
ax.set(xlabel='Unique donors', title='Figure 1. Available donors by region')
fig.tight_layout()
fig.savefig(OUT / 'figure_1_region_counts.png', dpi=200)
plt.show()


## 5. Table 2 and Figure 5 — Prediction performance
Each direction uses matched donors, an 80/20 development/test split, and training-only gene selection and model tuning. **Table 2** reports median gene-wise test R², 95% donor-bootstrap intervals, baseline-relative skill and error measures. **Figure 5** plots median test R² with its 95% bootstrap intervals. Different pairs may use different donors and genes, so comparisons are exploratory.


In [ ]:
def choose_genes(frame, n):
    # Select adequately expressed, variable genes from training donors only.
    expressed = (frame > 1).mean(axis=0) >= 0.2
    variance = frame.var(axis=0)
    ranked = variance[expressed & (variance > 1e-8)].sort_values(ascending=False)
    if len(ranked) < 10:
        raise ValueError('Too few expressed, variable genes')
    return ranked.index[:n]


def analyze_pair(source, target):
    ids = regions[source].index.intersection(regions[target].index).sort_values()
    if len(ids) < MIN_PAIRED_DONORS:
        return None

    # Split by donor, never by gene or tissue sample.
    dev, test = train_test_split(np.arange(len(ids)), test_size=0.2, random_state=SEED)
    inner, val = train_test_split(dev, test_size=0.25, random_state=SEED)
    Xframe = regions[source].loc[ids]
    Yframe = regions[target].loc[ids]
    xgenes = choose_genes(Xframe.iloc[inner], N_SOURCE_GENES)
    ygenes = choose_genes(Yframe.iloc[inner], N_TARGET_GENES)
    X = Xframe[xgenes].to_numpy(dtype='float64')
    Y = Yframe[ygenes].to_numpy(dtype='float64')

    # Tune Ridge alpha using only training/validation donors.
    scaler = StandardScaler().fit(X[inner])
    Xi, Xv = scaler.transform(X[inner]), scaler.transform(X[val])
    validation_errors = []
    for alpha in ALPHAS:
        model = Ridge(alpha=alpha, solver='cholesky').fit(Xi, Y[inner])
        validation_errors.append(np.mean((Y[val] - model.predict(Xv))**2))
    best_alpha = ALPHAS[int(np.argmin(validation_errors))]

    # Refit on development donors; evaluate once on held-out test donors.
    scaler = StandardScaler().fit(X[dev])
    model = Ridge(alpha=best_alpha, solver='cholesky').fit(scaler.transform(X[dev]), Y[dev])
    truth = Y[test]
    pred = model.predict(scaler.transform(X[test]))
    baseline = np.broadcast_to(Y[dev].mean(axis=0), truth.shape)
    model_loss = ((truth - pred)**2).mean(axis=1)
    baseline_loss = ((truth - baseline)**2).mean(axis=1)
    skill = 1 - model_loss.mean() / baseline_loss.mean()

    # R² for each target gene in unseen donors.
    sse = ((truth - pred)**2).sum(axis=0)
    sst = ((truth - truth.mean(axis=0))**2).sum(axis=0)
    gene_r2 = 1 - np.divide(sse, sst, out=np.full_like(sse, np.nan), where=sst > 1e-12)

    # Bootstrap test donors; predictions and fitted model are held fixed.
    rng = np.random.default_rng(SEED)
    samples = rng.integers(0, len(test), size=(N_BOOTSTRAPS, len(test)))
    boot_r2 = []
    for indices in samples:
        t, p = truth[indices], pred[indices]
        den = ((t - t.mean(axis=0))**2).sum(axis=0)
        gene_scores = 1 - np.divide(
            ((t - p)**2).sum(axis=0), den,
            out=np.full_like(den, np.nan), where=den > 1e-12
        )
        boot_r2.append(np.nanmedian(gene_scores))
    r2_low, r2_high = np.nanquantile(boot_r2, [0.025, 0.975])

    return dict(
        source=source, target=target,
        n_paired=len(ids), n_test=len(test),
        median_gene_r2=float(np.nanmedian(gene_r2)),
        r2_ci_low=float(r2_low), r2_ci_high=float(r2_high),
        skill=float(skill),
        model_mae=float(np.mean(np.abs(truth - pred))),
        model_rmse=float(np.sqrt(model_loss.mean())),
        alpha=best_alpha,
    )


pairs = (list(itertools.permutations(active, 2)) if RUN_ALL_PAIRS
         else [(a, b) for a, b in ILLUSTRATIVE_PAIRS if a in active and b in active])
rows = []
for source, target in pairs:
    row = analyze_pair(source, target)
    if row is not None:
        rows.append(row)
        print(f"{source} → {target}: median R²={row['median_gene_r2']:.3f}")
if not rows:
    raise ValueError('No region pairs met the minimum paired-donor threshold')

# Table 2. Held-out-donor performance by prediction direction.
results = pd.DataFrame(rows)
results.to_csv(OUT / 'table_2_prediction_results.csv', index=False)
display(results)

# Figure 5. Gene-wise median test R² with 95% donor-bootstrap intervals.
r = results.sort_values('median_gene_r2')
y = np.arange(len(r))
fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(r))))
ax.hlines(y, r.r2_ci_low, r.r2_ci_high, color='#357da8', linewidth=2.5)
ax.scatter(r.median_gene_r2, y, color='#163b54', s=50, zorder=3)
ax.set_yticks(y, [a.replace('_', ' ') + ' → ' + b.replace('_', ' ')
                  for a, b in zip(r.source, r.target)])
ax.axvline(0, color='gray', linestyle='--')
ax.set(xlabel='Median gene-wise test R²', title='Figure 5. Cross-region prediction R² (95% intervals)')
fig.tight_layout()
fig.savefig(OUT / 'figure_5_gene_r2.png', dpi=200)
plt.show()
